# ShopDesk, Section 1 Lab 1: Build Your First Agent

A beginner-friendly notebook built on the **base Anthropic SDK**, running **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**. We build the smallest possible
ShopDesk agent from scratch: one tool, one call, one `tool_use` to `tool_result`
round-trip. This is the foundation the agentic loop (next section) is built on.

## The real-world scenario

A plain prompt can write a nice reply, but it cannot *know* whether order A1 has shipped;
that fact lives in ShopDesk's data, not in the model. An **agent** closes that gap: it is
an LLM that reasons about a goal and then **acts** on its environment through tools. Here
the goal is a customer's question, and the one tool is a lookup into the order book.

The question this lab answers: **what is the minimum machinery that turns a chat model
into an agent that can look something up and use the answer?**

## Objectives

- Name the three building blocks every agent needs: an **LLM** (the reasoning engine),
  a **tool** (a function it can call to act), and a **prompt** (the goal that steers it).
- Define one tool with a **JSON schema** so the model knows how to call it.
- Wire the LLM call and handle a single **tool_use to tool_result** round-trip end to end.
- See the contrast with a single-shot prompt, and learn when an agent is the right choice.

## What you'll observe

- A single-shot call (no tools) cannot state A1's real status; it can only guess.
- With the tool wired in, the model's `stop_reason` comes back as `tool_use`: the decision
  to act.
- After we run the tool and return the result, the model answers with the real status and
  `stop_reason` flips to `end_turn`.

## How to run

Run top to bottom. The tool, the schema, and the concept cells are pure Python and run
anywhere. The two cells that call Claude are live: paste a real key into **Setup 2/3** and
re-run from the top, otherwise they skip cleanly.

## 0. Setup

**This cell:** installs the packages. This lab uses only the **base Anthropic SDK**,
because an agent is just a Messages API call with a `tools` list plus a little code around
it. We install once so every cell below can import.

In [ ]:
# ===== SETUP 1/3 - install the base SDK =====
%pip install -q anthropic python-dotenv

**This cell:** imports what we need, pins the model, and sets a `RUN_LIVE` switch so
the two model calls fire only when a real key is present. We build the switch now so each
live cell can guard itself and still run offline.

In [ ]:
# ===== SETUP 2/3 - imports, the model, and a live/offline switch =====
import os                                       # read the API key from the environment
import json                                     # print tool payloads readably
import anthropic                                # the base Anthropic SDK (synchronous)

try:                                            # load a .env file if present (keeps keys out of code)
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # python-dotenv not installed? that is fine
    pass                                        #   the key can be set another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds the shared **ShopDesk world**: two orders and their statuses.
This is the environment the agent acts on; the tool we write next simply reads from it.

In [ ]:
# ===== SETUP 3/3 - the shared ShopDesk data =====
ORDERS = {                                       # our tiny order book (the agent's environment)
    "A1": {"status": 2},                         #   A1 has shipped
    "A2": {"status": 3},                         #   A2 was delivered
}
STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}   # status code -> human word
print("orders:", list(ORDERS))                   # quick sanity check

### What is an agent?

A **single-shot prompt** takes text in and gives text out; it cannot reach outside itself.
A **hardcoded pipeline** runs fixed steps in a fixed order with no decisions. An **agent**
sits between them: an LLM that **reasons** about a goal and **acts** through tools, deciding
for itself whether to answer directly or call a tool first.

Every agent is made of three parts:

- **LLM** - the reasoning engine that decides what to do next.
- **Tools** - functions it can call to act on the world (here, look up an order).
- **Prompt** - the role, instructions, and goal that steer it. In this minimal lab the
  prompt is just the customer's question; the next lab adds a system prompt.

---

### 🎯 Lab objective - the smallest working agent

**What you build:** one tool, its JSON schema, and the code that handles a single
`tool_use` to `tool_result` round-trip, so the model can look up a real status and use it.

**Why it helps you build real solutions:** every agent framework, however large, is this
round-trip repeated. Build it once by hand and the rest is just scaling it up.

**How you'll see it:** the model asks to call the tool (`stop_reason` is `tool_use`), you
run it, hand back the result, and the model answers with the real status (`end_turn`).

**This cell:** building block 1, the **tool body**: a plain Python function that
reads the order book. This is the *action* the agent can take. It is ordinary code; the
model never runs it, it only asks us to.

In [ ]:
# ===== the tool body (plain Python) =====
def get_order_status(order_id):                  # the one action our agent can take
    o = ORDERS.get(order_id)                     #   look the order up in the environment
    if not o:                                    #   no such order?
        return "unknown order"                   #     say so plainly
    return STATUS_NAMES[o["status"]]             #   return the human-readable status word

RUN_TOOL = {"get_order_status": get_order_status}   # name -> function, so we can dispatch by name
print(get_order_status("A1"))                    # sanity check: should print 'shipped'

**This cell:** the **JSON schema** for that tool. This, not the Python body, is what
the model reads to decide when and how to call the tool. The `name` and `description` help
it choose; the `input_schema` tells it the one argument to supply.

In [ ]:
# ===== the tool schema the model reads =====
TOOLS = [                                         # the list of tools handed to the model
    {
        "name": "get_order_status",              #   must match the key in RUN_TOOL
        "description": "Look up the current status of an order by its id.",   # helps the model choose
        "input_schema": {                        #   the shape of the tool's input
            "type": "object",                    #     an object...
            "properties": {                      #     ...with named fields
                "order_id": {"type": "string"}   #       one string field
            },
            "required": ["order_id"],            #     which is required
        },
    }
]
print("tools defined:", [t["name"] for t in TOOLS])   # confirm the tool is registered

**This cell:** the **single-shot baseline**, no tools attached. We ask the model the
same question with no way to look anything up, so it can only guess or refuse. This is the
gap the agent will close.

In [ ]:
# ===== single-shot: no tools, so the model cannot know =====
QUESTION = "What is the current status of order A1?"   # the goal, stated once

if RUN_LIVE:                                      # needs a real key
    client = anthropic.Anthropic()                #   the LLM client (reads the key)
    r = client.messages.create(model=MODEL, max_tokens=256,   # a plain call, NO tools
                               messages=[{"role": "user", "content": QUESTION}])
    print("".join(b.text for b in r.content if b.type == "text"))   # it can only guess
else:
    print("[skipped - set ANTHROPIC_API_KEY to run this live]")

**This cell:** the **round-trip helper**, the heart of the lab. It does four steps:
(1) the model decides and returns `tool_use`, (2) we read which tool it wants, (3) we run
that Python function, (4) we hand the result back so the model can answer. This is a single
round-trip: exactly one tool call, then the final reply.

In [ ]:
# ===== the agent: one tool_use -> tool_result round-trip =====
def run_once(question):                           # take a goal, use the tool once, then answer
    client = anthropic.Anthropic()                #   the LLM (building block 1)
    messages = [{"role": "user", "content": question}]   # the prompt here is just the goal

    first = client.messages.create(model=MODEL, max_tokens=512,   # STEP 1: the model decides
                                   tools=TOOLS, messages=messages)
    print("decision (stop_reason):", first.stop_reason)   #   'tool_use' means it wants to act
    if first.stop_reason != "tool_use":           #   if it did not ask for a tool...
        return "".join(b.text for b in first.content if b.type == "text")   # ...just return its text

    call = next(b for b in first.content if b.type == "tool_use")   # STEP 2: read the tool call
    print("  -> calls", call.name, "with", call.input)             #   which tool, which input
    result = RUN_TOOL[call.name](**call.input)     # STEP 3: run the real Python function
    print("  -> tool result:", result)             #   the fact we got back

    messages.append({"role": "assistant", "content": first.content})   # keep the model's turn
    messages.append({"role": "user", "content": [  # STEP 4: hand the result back as a tool_result
        {"type": "tool_result", "tool_use_id": call.id, "content": result}]})

    second = client.messages.create(model=MODEL, max_tokens=512,   # the model now writes the answer
                                    tools=TOOLS, messages=messages)
    print("final (stop_reason):", second.stop_reason)              # expect 'end_turn'
    return "".join(b.text for b in second.content if b.type == "text")   # the grounded reply

**This cell:** runs the agent on the same question. Watch the two `stop_reason`
prints: `tool_use` first (the decision to act), then `end_turn` (the answer), with the
real status in between.

In [ ]:
# ===== run the agent =====
if RUN_LIVE:                                      # needs a real key
    print("ANSWER:", run_once(QUESTION))          #   the agent looks A1 up and reports it
else:
    print("[skipped - set ANTHROPIC_API_KEY to run this live]")

| anti-pattern | what to do instead |
|---|---|
| ask a plain prompt for live facts it cannot know | give it a tool and let it look the fact up |
| guess the tool from the reply text | branch on `stop_reason == "tool_use"`, not on prose |
| forget the `tool_use_id` on the result | echo the exact id so the model matches result to call |
| skip returning the tool_result | append it, or the model answers with nothing to go on |

### When is an agent the right choice?

An agent is not always the answer. It costs more (several model calls), adds latency, and
is less predictable than fixed code. Reach for the simplest thing that works:

| Situation | Better choice | Why |
|---|---|---|
| One fixed transformation of known input | a single prompt | cheapest, one call, deterministic |
| Fixed steps in a fixed order, no branching | a hardcoded pipeline | reliable, no per-step model call |
| Goal needs live data or genuine decisions | an agent | it reasons and calls tools as needed |

Use an agent when the task truly needs *reasoning plus action*; use a prompt or a pipeline
when it does not.

**Lesson:** an agent is an **LLM** plus **tools** plus a **prompt**, wired so the
model can decide to act and then use what it learns. The whole mechanism is one round-trip:
decide (`tool_use`), run the tool, return the result, answer (`end_turn`). Everything after
this is that same loop, repeated and scaled.

---

## Recap - the three blocks and one round-trip

| Building block | In this lab | Course topic |
|---|---|---|
| LLM | the Sonnet call that decides | the reasoning engine |
| Tool | `get_order_status` + its JSON schema | functions the model can call |
| Prompt | the customer's question (goal only) | role, instructions, and goal |
| Round-trip | `tool_use` -> run tool -> `tool_result` -> `end_turn` | how an agent operates end to end |

One principle to carry forward: **the model reasons and decides; your code gives it tools
and runs them.** To run live, paste a real key into **Setup 2/3** and re-run from the top.
Then try it: ask about order A2, or an order that does not exist, and watch the tool result
change the answer. Next section adds a second tool and a system prompt, then the full loop.